In [ ]:
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter

In [ ]:
#@numba.jit
def Entropy(image):
    # Convert the image to grayscale using manual conversion
    gray_image = 0.299 * image[:, :, 2] + 0.587 * image[:, :, 1] + 0.114 * image[:, :, 0]
    # Calculate the histogram of the grayscale image
    hist, _ = np.histogram(gray_image, bins=256, range=(0, 256), density=True)
    # Avoid using SciPy's entropy function, instead calculate it manually
    hist = hist[hist > 0]  # Remove zero entries to avoid log(0)
    hist_entropy = -np.sum(hist * np.log2(hist))
    return hist_entropy

In [ ]:
def relative_luminance(image):
    b, g, r = cv2.split(image)
    luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
    #return np.mean(luminance)
    return luminance.sum()/(image.shape[0]*image.shape[1]*255)

def color_variance(image):
    variance_b = np.var(image[:, :, 0])
    variance_g = np.var(image[:, :, 1])
    variance_r = np.var(image[:, :, 2])
    return ((variance_b+variance_g+variance_r)/3)/(image.shape[0]*image.shape[1])

def calculate_variability(image,kernel_size=3):
    kernel = np.ones((kernel_size,kernel_size))/(kernel_size**2-1)
    kernel[kernel_size//2,kernel_size//2] = -1
    V = cv2.filter2D(image,-1,kernel)
    return abs(V).sum()/image.size

In [ ]:
def canny_edge(image,tl=0,th=150,inverted=True):
    # Convert to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Apply Canny edge detector
    edges = cv2.Canny(gray_image, threshold1=tl, threshold2=th)
    # Invert the binary image (0 becomes 255, and 255 becomes 0)
    if inverted:
        edges = cv2.bitwise_not(edges)
    return edges.astype(np.uint8)

In [ ]:
def sobel_edge(image,tl=50,th=255):
    # Convert to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Compute the gradients using the Sobel operator
    sobelx = cv2.Sobel(gray_image, cv2.CV_64F, 1, 0, ksize=3)  # Gradient in x-direction
    sobely = cv2.Sobel(gray_image, cv2.CV_64F, 0, 1, ksize=3)  # Gradient in y-direction
    # Calculate the magnitude of the gradient
    magnitude = np.sqrt(sobelx**2 + sobely**2)
    # Normalize to range 0 to 255 and convert to uint8
    magnitude = np.uint8(255 * magnitude / np.max(magnitude))
    # Convert to binary image (0 or 255) and invert it
    _, binary_edges = cv2.threshold(magnitude, tl, th, cv2.THRESH_BINARY_INV)
    return binary_edges

In [ ]:
def scharr_edge(image,tl=50,th=255):
    # Convert to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Compute the gradients using the Scharr operator
    scharrx = cv2.Scharr(gray_image, cv2.CV_64F, 1, 0)  # Gradient in x-direction
    scharry = cv2.Scharr(gray_image, cv2.CV_64F, 0, 1)  # Gradient in y-direction
    # Calculate the magnitude of the gradient
    magnitude = np.sqrt(scharrx**2 + scharry**2)
    # Normalize to range 0 to 255 and convert to uint8
    magnitude = np.uint8(255 * magnitude / np.max(magnitude))
    # Convert to binary image (0 or 255) and invert it
    _, binary_edges = cv2.threshold(magnitude, tl, th, cv2.THRESH_BINARY_INV)
    return binary_edges

In [ ]:
def local_entropy(image, window_size=9):
    if window_size % 2 == 0:
        raise ValueError("window_size must be odd")
    if image.ndim == 3:
        channels = [local_entropy(image[:, :, c], window_size) for c in range(image.shape[2])]
        return np.stack(channels, axis=-1)
    pad = window_size // 2
    padded = np.pad(image, pad, mode='reflect')
    max_bits = np.log2(window_size**2)
    h, w = image.shape
    entropy_img = np.zeros((h, w), dtype=np.float32)
    gray_levels = 256
    for i in range(h):
        for j in range(w):
            window = padded[i:i+window_size, j:j+window_size]
            hist = np.bincount(window.ravel(), minlength=gray_levels)
            probs = hist / hist.sum()
            probs = probs[probs > 0]
            entropy = -np.sum(probs * np.log2(probs))
            entropy_img[i, j] = entropy
    rescaled = (entropy_img / max_bits) * 255
    return rescaled.astype(np.uint8)

In [ ]:
def local_ssim(img1, img2, win_size= 7):
    if img1.shape != img2.shape or img1.ndim != 3 or img1.shape[2] != 3:
        raise ValueError("Both inputs must have shape (H, W, 3) and match in size.")
    ssim_map_all = np.zeros_like(img1, dtype=np.float32)
    for c in range(3):
        _, ssim_map = ssim(
            img1[:, :, c],
            img2[:, :, c],
            win_size=win_size,
            data_range=img1[:, :, c].max() - img1[:, :, c].min(),
            channel_axis=None,
            full=True
        )
        ssim_map_all[:, :, c] = ssim_map
    return ssim_map_all

In [ ]:
def local_psnr(img1: np.ndarray, img2: np.ndarray, win_size: int = 7, max_psnr: float = 255.0) -> np.ndarray:
    if img1.shape != img2.shape or img1.ndim != 3 or img1.shape[2] != 3:
        raise ValueError("Both inputs must have shape (H, W, 3) and match in size.")
    img1 = img1.astype(np.float32)
    img2 = img2.astype(np.float32)
    data_range = np.zeros(3)
    for c in range(3):
        data_range[c] = img1[:,:,c].max() - img1[:,:,c].min()
    psnr_map_all = np.zeros_like(img1, dtype=np.float32)
    for c in range(3):
        diff = img1[:, :, c] - img2[:, :, c]
        mse_map = uniform_filter(diff**2, size=win_size, mode='reflect')
        with np.errstate(divide='ignore'):
            psnr_map = 10 * np.log10((data_range[c]**2) / mse_map)
        psnr_map[~np.isfinite(psnr_map)] = max_psnr
        psnr_map_all[:, :, c] = psnr_map
    return 1/(psnr_map_all+1e-10)

In [ ]:
def local_covariance_map(frameA, frameB, win=3):
    assert frameA.shape == frameB.shape, "Frames must have the same size"
    # Clean inputs (replace NaN and Inf with 0)
    A = np.nan_to_num(frameA.astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    B = np.nan_to_num(frameB.astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    n = win * win
    k = np.ones((win, win), dtype=np.float64)
    sumA  = cv2.filter2D(A, -1, k, borderType=cv2.BORDER_REFLECT)
    sumB  = cv2.filter2D(B, -1, k, borderType=cv2.BORDER_REFLECT)
    sumAB = cv2.filter2D(A * B, -1, k, borderType=cv2.BORDER_REFLECT)
    meanA = sumA / n
    meanB = sumB / n
    cov_map = (sumAB / n) - (meanA * meanB)
    return cov_map.astype(np.float32)

In [ ]:
def chromaticity(img1,eps=1e-6):
    img1 = img1.astype(np.float32) + eps
    sum1 = img1.sum(axis=2, keepdims=True)
    chroma1 = img1 / sum1
    return chroma1

In [ ]:
def chromaticity_difference(img1, img2, eps=1e-6):
    img1 = img1.astype(np.float32) + eps
    img2 = img2.astype(np.float32) + eps
    sum1 = img1.sum(axis=2, keepdims=True)
    sum2 = img2.sum(axis=2, keepdims=True)
    chroma1 = img1 / sum1
    chroma2 = img2 / sum2
    dist = np.linalg.norm(chroma2 - chroma1, axis=2)
    return dist

In [ ]:
def likelihood_maps_video(frames, patch_size=5):
    # Gaussian smoothed patch-based likelihood
    gray_frames = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) if f.ndim==3 else f.copy() for f in frames]
    all_pixels = np.concatenate([g.flatten() for g in gray_frames])
    hist, bin_edges = np.histogram(all_pixels, bins=256, range=(0,256), density=True)
    prob_distribution = hist / hist.sum()
    likelihood_maps = []
    for gray in gray_frames:
        likelihood = prob_distribution[gray]
        # Apply small Gaussian smoothing to reduce noise
        likelihood_maps.append(gaussian_filter(likelihood.astype(np.float32), sigma=1))
    return likelihood_maps

In [ ]:
def multi_gabor_kernels(sizes=[(3,3),(5,5),(10,10)],scales=[1,3],orientations=[0,np.pi/4,np.pi/2,3*np.pi/4],freq=[2,3,5],phase=[0]):
    kernels = []
    for s in scales:
        for theta in orientations:
            for size in sizes:
                for f in freq:
                    for p in phase:
                        k = cv2.getGaborKernel(size, sigma=s, theta=theta, lambd=f, gamma=0.5, psi=p)
                        kernels.append(k)
    return kernels

In [ ]:
def apply_gabbor_filters(img,sizes=[(3,3),(5,5),(10,10)],scales=[1,3],orientations=[0,np.pi/4,np.pi/2,3*np.pi/4],freq=[2,3,5],phase=[0]):
    filtered_images = []
    kernels = multi_gabor_kernels(sizes,scales,orientations,freq)
    for k in kernels:
        f_img = cv2.filter2D(img, cv2.CV_32F, k)
        filtered_images.append(f_img)
    return filtered_images

In [ ]:
def get_gabbor_dif(Frames):
    Kernels = multi_gabor_kernels()
    Dif_Gab = []
    for t in range(len(Frames)-1):
        D = []
        for k in Kernels:
            f_img0 = cv2.filter2D(Frames[t], cv2.CV_32F, k)
            f_img1 = cv2.filter2D(Frames[t+1], cv2.CV_32F, k)
            D.append(abs(f_img0-f_img1))
        Dif_Gab.append(np.mean(D,axis=0))
    return Dif_Gab